# 13.2 偏好資料怎麼表示？

# 第 13 章：把回答偏好變成訓練

前置：第7章SFT，以及第8–9章的「好回答」規則。可以直接走這條路，不必先學圖片。把兩篇答案像兩道菜一起試喫：喜歡哪一篇，與「兩篇是否都安全」是不同問題。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：試喫兩道菜：比起原廚師，更偏向好回答**

chosen 與 rejected 必須對同一個問題。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/dpo.svg")))

**先想一想：**兩篇回答其實來自不同問題，還能直接學偏好嗎？

prompt、chosen、rejected是一個三件組。兩篇回答必須對同一題與同一條件，不是任意拿一篇好文和一篇差文拼在一起。

**把直覺寫成數學：**shared prompt+two continuations；內容ID與來源保持對應。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.data import render_chat

prompt = "1+1=?"
chosen = "2"
rejected = "3"
for answer in (chosen, rejected):
    x, y = render_chat([{"role": "user", "content": prompt}, {"role": "assistant", "content": answer}])
    print(x.tolist(), y.tolist())

**如何讀結果：**同樣prompt token和role邊界；分數只加答案位置，問題不算。

**只改一件事：**只換rejected的題目，指出資料對不成立。
